# model_ridge

Notebook generated by automatic conversion. The code keeps comments in English and can be executed from the `notebooks/ml-models` folder.

## Import necessary libraries

In [6]:
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import RidgeCV, Ridge as SklearnRidge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from pipeline import load_and_preprocess, DATA_FOLDER

## Data loading and preprocessing

In [7]:
os.makedirs("outcome", exist_ok=True)

# Load and preprocess data using shared pipeline
X_train, X_val, X_test, y_train, y_val, y_test, feature_names = load_and_preprocess(DATA_FOLDER)
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
X_val = X_val.astype(np.float32)

Train shape: (5170, 17249)
Val shape: (738, 17249)
Test shape: (1477, 17249)


## Ridge model configuration

In [8]:
# Try to use the GPU version of cuML Ridge, otherwise sklearn
use_gpu = False
try:
    from cuml.linear_model import Ridge as CumlRidge

    model = CumlRidge(alpha=1.0)
    use_gpu = True
    chosen_alpha = 1.0
except Exception:
    warnings.warn("cuML Ridge not available, using sklearn RidgeCV on CPU.")
    model = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=None)

/tmp/ipykernel_25874/286409907.py:10: UserWarning: cuML Ridge not available, using sklearn RidgeCV on CPU.
  warnings.warn("cuML Ridge not available, using sklearn RidgeCV on CPU.")


## Model training

In [9]:
model.fit(X_train, y_train)

if not use_gpu:
    chosen_alpha = float(model.alpha_)

## Evaluation and saving

In [10]:
preds = model.predict(X_test)
preds = np.asarray(preds).ravel()
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)
print(f"[Ridge] RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")
print(f"[Ridge] Chosen alpha: {chosen_alpha}")

# Predicted vs Real plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, preds, alpha=0.6, color='purple', edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel('Real Values')
plt.ylabel('Predicted Values')
plt.title('Ridge: Predicted vs Real')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("outcome/ridge_pred_vs_real.png", dpi=300, bbox_inches='tight')
plt.close()

# Top-10 coefficients by absolute importance
coef = np.asarray(model.coef_).ravel()
top_indices = np.argsort(np.abs(coef))[::-1][:10]
top_coefs = [(int(i), float(coef[i]), feature_names[i] if i < len(feature_names) else f"feature_{i}") for i in top_indices]
print("[Ridge] Top-10 coefficients (index, coeff, feature name):")
for idx, value, name in top_coefs:
    print(f"  {idx}: {name} = {value:.6f}")

# Coefficients plot
plt.figure(figsize=(12, 6))
top_indices_plot = top_indices[:10]  # Top 10 for plotting
top_coefs_plot = coef[top_indices_plot]
plt.barh(range(len(top_indices_plot)), top_coefs_plot, color='mediumpurple', edgecolor='black')
plt.yticks(range(len(top_indices_plot)), [feature_names[i] if i < len(feature_names) else f"feature_{i}" for i in top_indices_plot])
plt.xlabel('Coefficient Value')
plt.title('Ridge Top-10 Feature Coefficients')
plt.grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs("outcome/feature_importance", exist_ok=True)
plt.savefig("outcome/feature_importance/ridge_coefficients.png", dpi=300, bbox_inches='tight')
plt.close()

# Save model
os.makedirs("outcome/models", exist_ok=True)
joblib.dump(model, "outcome/models/ridge_model.pkl")

print(f"[Ridge] Model and plots saved to outcome/ folder")

[Ridge] RMSE: 0.0086 | MAE: 0.0050 | R²: 1.0000
[Ridge] Chosen alpha: 0.01
[Ridge] Top-10 coefficients (index, coeff, feature name):
  1434: num__mordred_ETA_eta_B = -0.112735
  1436: num__mordred_ETA_eta_BR = -0.094369
  1004: num__mordred_Xc-3d = -0.067502
  453: num__mordred_ATSC1d = 0.050535
  1020: num__mordred_Xp-2d = -0.042883
  1006: num__mordred_Xc-5d = -0.037246
  1810: num__mordred_mZagreb1 = 0.036665
  1500: num__mordred_Kier1 = 0.035128
  1014: num__mordred_Xpc-6d = -0.033510
  1005: num__mordred_Xc-4d = 0.033446
[Ridge] Model and plots saved to outcome/ folder
